In [2]:
import sys
import os
from pathlib import Path

if 'google.colab' in sys.modules:
    print("Ambiente Colab rilevato. Inizializzazione in corso...")
    
    # 1. Monta Drive (se non già montato)
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
        
    # Imposta la root del Drive su Colab
    BASE_DIR = Path('/content/drive/Shareddrives/Progetti/Progetto_MLDM')

    GIT_BRANCH = 'zone-test'
    
    # 2. Clona/Aggiorna la repo
    REPO_DIR = '/content/crop-spatial-classification'
    if not os.path.exists(REPO_DIR):
        !git clone -b {GIT_BRANCH} https://github.com/SimoRinaldi/crop-spatial-classification.git {REPO_DIR}
    else:
        !cd {REPO_DIR} && git fetch origin && git checkout -B {GIT_BRANCH} origin/{GIT_BRANCH} && git reset --hard origin/{GIT_BRANCH} && git clean -fd
    
    # 3. Aggancia la cartella per permettere gli import
    if REPO_DIR not in sys.path:
        sys.path.append(REPO_DIR)
        
    # 4. Installa i requisiti in modo silenzioso
    %pip install -q -r {REPO_DIR}/requirements.txt
    print("Setup ambiente Colab completato! Branch attivo su Colab: ", GIT_BRANCH)
    
else:
    print("Ambiente Locale rilevato. Procedo con l'esecuzione...")
    
    BASE_DIR = Path.cwd().parent

    if str(BASE_DIR) not in sys.path:
        sys.path.append(str(BASE_DIR))

# =============================
# DEFINIZIONE PATH UNIVERSALI
# =============================
DATA_DIR = BASE_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
INTERIM_DIR = DATA_DIR / 'interim'
PROCESSED_DIR = DATA_DIR / 'processed'

# Test di verifica per assicurarsi che i dati siano accessibili
if RAW_DIR.exists():
    print(f"✅ Collegamento ai dati riuscito! Cartella raw: {RAW_DIR}")
else:
    print(f"❌ Attenzione: Cartella non trovata in {RAW_DIR}. Verifica il nome, il mount o il path locale.")
     


Mounted at /content/drive
Cloning into '/content/crop-spatial-classification'...
remote: Enumerating objects: 859, done.
remote: Counting objects: 100% (277/277), done.
remote: Compressing objects: 100% (108/108), done.
remote: Total 859 (delta 220), reused 196 (delta 169), pack-reused 582 (from 1)
Receiving objects: 100% (859/859), 197.19 MiB | 32.06 MiB/s, done.
Resolving deltas: 100% (594/594), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 87.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 9.7 MB/s eta 0:00:00
Setup ambiente Colab completato! Branch attivo su Colab:  zone-test
✅ Collegamento ai dati riuscito! Cartella raw: /content/drive/Shareddrives/Progetti/Progetto_MLDM/data/raw


# 01 - Estrazione del ground truth e campionamento delle colture
estrae un campione bilanciato di punti per ciascun tipo di coltura e genera la legenda associata.

#### Ricerca dei file TIF per ciascun anno

In [3]:
from pathlib import Path
from src.config import YEARS_TO_FETCH

# anno/i di estrazione dei dati
TARGET_YEARS = YEARS_TO_FETCH

main_directory = Path(f"{DATA_DIR}/raw/zone_test")
tifs_3035 = {}



for year in TARGET_YEARS:
    year_dir = main_directory / year
    if year_dir.is_dir():
        # trova tutti i .tif per questo anno
        tifs_3035[year] = [str(tif) for tif in year_dir.rglob("*.tif")]
        print(f"Anno {year}: trovati {len(tifs_3035[year])} file .tif")
    else:
        print(f"Cartella anno {year} non trovata in {main_directory}")

Anno 2023: trovati 1 file .tif
Anno 2022: trovati 1 file .tif
Anno 2021: trovati 1 file .tif


In [1]:
main_directory = Path(f"{DATA_DIR}/raw/zone_test")

for item in sorted(main_directory.iterdir()):
      if item.is_dir():  # controlla che sia una cartella e non un file
        nome_cartella = item.name
        print(nome_cartella)


NameError: name 'Path' is not defined

#### Riproiezione in coordinate GPS (EPSG:4326)

In [4]:
from pathlib import Path
import rioxarray
from rasterio.enums import Resampling

tifs_4326 = {}

for year, file_paths in tifs_3035.items():
    year_file_list = []

    for row_path in file_paths:
        # costruisce il percorso del file riproiettato in EPSG:4326
        file_name = row_path.replace("03035", "4326").replace("raw", "processed")
        file_path = Path(file_name)
        
        # se il file riproiettato esiste già, salta la riproiezione
        if file_path.is_file():
            year_file_list.append(file_name)
            continue
        
        # crea la cartella di destinazione se non esiste
        file_path.parent.mkdir(parents=True, exist_ok=True)
        
        # apre il file GeoTIFF originale (EPSG:3035)
        raster_3035 = rioxarray.open_rasterio(row_path)

        # riproiezione in EPSG:4326, usando NEAREST per preservare i codici delle colture
        raster_4326 = raster_3035.rio.reproject("EPSG:4326", resampling=Resampling.nearest)

        # salva il file riproiettato
        raster_4326.rio.to_raster(file_name)
        raster_3035.close()

        year_file_list.append(file_name)
        
    tifs_4326[year] = year_file_list

print(f"\nRiproiezione completata per {sum(len(v) for v in tifs_4326.values())} file")


Riproiezione completata per 3 file


#### Creazione e salvataggio legenda delle colture

In [10]:
import pandas as pd
import xml.etree.ElementTree as ET

from src.config import EXCLUDED_CROP_CLASSES

# cerca i files .aux.xml che contengono i metadati
files = list(main_directory.rglob("*.aux.xml"))

if not files:
    raise FileNotFoundError(f"Nessun file .aux.xml trovato all'interno di: {main_directory}")

# prende il primo file trovato
file_path = files[0]
print(f"Trovato file .aux.xml: {file_path.name}")

tree = ET.parse(file_path)
root = tree.getroot()

legend = {}

# estrae il codice e il nome della coltura
for r in root.findall(".//Row"):
    fields = r.findall("F")
    
    crop_code = int(fields[0].text)     

    if crop_code in EXCLUDED_CROP_CLASSES:
        continue
    
    crop_count = fields[1].text
    crop_name = fields[2].text

    legend[crop_code] = {"name": crop_name, "count": crop_count}

Trovato file .aux.xml: CLMS_HRLVLCC_CTY_S2024_R10m_E47N21_03035_V01_R00.tif.aux.xml


#### Riepilogo del campione di colture

In [11]:
import pandas as pd

df_legend = (
        pd.DataFrame.from_dict(legend, orient="index")
        .rename_axis("crop_code")
        .reset_index()
    )
  
display(df_legend)

,crop_code,name,count
0,1110,Wheat,1602002
1,1120,Barley,670789
2,1130,Maize,12332
3,1140,Rice,188
4,1150,Other Cereals,39137
5,1210,Fresh Vegetables,494150
6,1220,Dry Pulses,531316
7,1310,Potatoes,10912
8,1320,Sugar Beet,0
9,1410,Sunflower,340336
